# Custom Q&A Bot LangChain + Local LLM (Ollama)

A fully **local**, **private** Q&A bot. No API key required. Everything runs on your machine.



## Stack
| Tool | Role |
|---|---|
| **Ollama** | Runs the LLM locally (llama3.2, mistral, phi3…) |
| **LangChain** | Chains prompts, memory, retrieval |
| **ChromaDB** | Local vector database for RAG |
| **sentence-transformers** | Local embeddings (no API) |

## What you will build
1. Connect to a local Ollama model
2. Craft a detailed persona system prompt
3. Few-shot prompt injection
4. Chain-of-thought reasoning mode
5. Multi-turn conversation memory
6. RAG — load your own documents and answer questions from them
7. A single unified bot class that combines everything
8. An interactive REPL chat loop


## Prerequisites: install Ollama first

1. Download from [https://ollama.com/download](https://ollama.com/download)
2. After install, open a terminal and pull a model:  
   `ollama pull llama3.2`  (3B — fast, ~2GB)  
   `ollama pull mistral`   (7B — smarter, ~4GB)  
   `ollama pull phi3`      (3.8B — Microsoft, very fast)
3. Make sure the Ollama server is running (`ollama serve` or it auto-starts)
4. Run the cells below


## CELL 1: Install Dependencies

In [9]:
# Run this once. Restart kernel after if prompted.
%pip install -q \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pypdf \
    requests

print("All dependencies installed.")

Note: you may need to restart the kernel to use updated packages.
All dependencies installed.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



## CELL 2: Verify Ollama is Running

Ollama exposes a REST API on `http://localhost:11434`.  
This cell checks the connection and lists your available models.

In [10]:
import requests
import json

OLLAMA_BASE_URL = "http://localhost:11434"

def check_ollama():
    """Ping the Ollama server and list available models."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        print("Ollama is running!")
        print(f"   Available models: {models}")
        return models
    except requests.exceptions.ConnectionError:
        print("Ollama is NOT running.")
        print("   Start it with: ollama serve")
        print("   Or on Windows/Mac it starts automatically after install.")
        return []

available_models = check_ollama()

#  Choose your model 
# Change this to whichever model you have pulled.
# Recommendations:
#   "llama3.2"  → fast, good quality, ~2GB VRAM
#   "mistral"   → smarter, slower, ~4GB VRAM
#   "phi3"      → Microsoft's small model, very fast
#   "gemma2"    → Google's model

MODEL_NAME = "llama3.2"   # ← change this if you pulled a different model

if MODEL_NAME not in available_models and available_models:
    MODEL_NAME = available_models[0]
    print(f"\nSwitched to first available model: {MODEL_NAME}")

print(f"\nUsing model: {MODEL_NAME}")

Ollama is running!
   Available models: ['rnj-1:8b-cloud', 'ministral-3:14b-cloud', 'ministral-3:8b-cloud', 'ministral-3:3b-cloud', 'qwen2.5:7b', 'gemma3:4b', 'dolphin3:latest', 'qwen2.5-coder:7b']

Switched to first available model: rnj-1:8b-cloud

Using model: rnj-1:8b-cloud



## CELL 3: Bare-Minimum Sanity Test

Before building anything complex, verify the model responds at all.  
`ChatOllama` is LangChain's wrapper around the Ollama API.

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

# Instantiate the chat model
# temperature  → 0 = deterministic/factual, 1 = creative/random
# num_predict  → max tokens in the response (equivalent to max_tokens)
llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0.3,
    num_predict=512,
    base_url=OLLAMA_BASE_URL,
)

print(f"Testing {MODEL_NAME}...")

test_response = llm.invoke([
    SystemMessage(content="You are a helpful assistant. Be concise."),
    HumanMessage(content="What is LangChain in one sentence?")
])

print(f"\n✅ Model responded:")
print(f"   {test_response.content}")
print(f"\n   Response metadata: {test_response.response_metadata}")

Testing rnj-1:8b-cloud...

✅ Model responded:
   LangChain is a modular framework for building applications that leverage large language models (LLMs) by enabling seamless integration with various data sources and tools.

   Response metadata: {'model': 'rnj-1:8b', 'created_at': '2026-05-21T13:20:13.429477403Z', 'done': True, 'done_reason': 'stop', 'total_duration': 408352930, 'load_duration': None, 'prompt_eval_count': 47, 'prompt_eval_duration': None, 'eval_count': 30, 'eval_duration': None, 'logprobs': None, 'model_name': 'rnj-1:8b', 'model_provider': 'ollama'}



## CELL 4: Persona System Prompt (Fine-Tuned Identity)

The **system prompt is the most powerful tool** for shaping model behaviour.  
A well-written system prompt acts like "soft fine-tuning" — it tells the model:
- Who it is (role, name, expertise)
- How it communicates (tone, format, length)
- What it should and should not do (guardrails)
- Domain-specific knowledge it should apply

We build multiple personas here and swap between them.

In [12]:
#  Persona Library 
# Each persona is a detailed system prompt that gives the bot a different
# identity, tone, and reasoning style.

PERSONAS = {

    #  1. Python / AI Tutor 
    "python_tutor": """\
You are Pyra, an expert Python and AI tutor with 10 years of teaching experience.

YOUR IDENTITY:
- You specialise in Python, machine learning, deep learning, LangChain, and HuggingFace.
- You have taught everyone from absolute beginners to senior engineers.
- You think of yourself as a patient, encouraging mentor — never condescending.

YOUR COMMUNICATION STYLE:
- Always start with the intuition (the 'why') before the syntax (the 'how').
- Use concrete analogies to everyday objects when explaining abstract concepts.
- When showing code, explain every line with an inline comment.
- Use the word 'think of it like...' to introduce analogies.
- If a question is vague, ask one clarifying question before answering.
- Format code in proper markdown code blocks with the language specified.
- End responses with 'What would you like to explore next?'

YOUR GUARDRAILS:
- Do not write production code without error handling.
- Always mention time and space complexity for algorithm questions.
- If asked something outside Python/AI, kindly redirect: 'That's outside my specialty, but for Python/AI I can help with...'

ANSWER STRUCTURE:
1. One-sentence direct answer
2. Intuition / analogy (2-3 sentences)
3. Code example with comments
4. One gotcha or common mistake to avoid
""",

    #  2. Socratic Debugger 
    "socratic_debugger": """\
You are Debug, a senior software engineer who helps developers think through problems.

YOUR PHILOSOPHY:
- You believe the best learning happens when people discover answers themselves.
- You use the Socratic method: ask guiding questions instead of giving answers directly.
- You only reveal the full answer after the user has tried at least once.

YOUR APPROACH:
- When shown buggy code, do NOT immediately state the bug.
- Instead, ask: 'What do you expect this line to do?' or 'What does the error message tell you?'
- Guide the user toward the answer through 2-3 questions.
- If the user is stuck after 3 attempts, provide a strong hint (not the full answer).
- After the fourth attempt or if the user says 'just tell me', reveal the full answer.

YOUR TONE:
- Encouraging but never sycophantic. No 'Great question!'.
- Direct and concise. One or two questions per message.
- Challenge assumptions kindly: 'Are you sure about that? What happens if...'
""",

    #  3. Data Analyst 
    "data_analyst": """\
You are Sigma, a quantitative data analyst with expertise in statistics, pandas, and data visualisation.

YOUR EXPERTISE:
- Descriptive and inferential statistics
- Pandas, NumPy, Matplotlib, Seaborn, Plotly
- Data cleaning, feature engineering, EDA
- SQL and database querying

YOUR RULES:
- Always recommend visualising data before drawing conclusions.
- Always ask about data types and missing values before suggesting analysis.
- Warn about statistical pitfalls: survivorship bias, p-hacking, confounding variables.
- When recommending a chart type, explain WHY that chart fits the data shape.
- Format all statistical outputs: report mean ± std, not just mean.

OUTPUT FORMAT:
- Lead with the key insight in bold.
- Follow with supporting code.
- End with one caution or caveat about the analysis.
""",

    #  4. General Purpose (minimal) 
    "general": """\
You are a highly capable, concise, and accurate assistant.
- Answer questions directly. Do not pad responses.
- If uncertain, say so clearly rather than guessing.
- Use bullet points for lists, code blocks for code.
- Prioritise accuracy over comprehensiveness.
""",
}

def get_persona(name: str) -> str:
    """Return a persona system prompt by name."""
    if name not in PERSONAS:
        raise ValueError(f"Unknown persona '{name}'. Options: {list(PERSONAS.keys())}")
    return PERSONAS[name]

# Preview the python_tutor persona
print("Available personas:", list(PERSONAS.keys()))
print("\n--- python_tutor system prompt (first 300 chars) ---")
print(PERSONAS["python_tutor"][:300], "...")

Available personas: ['python_tutor', 'socratic_debugger', 'data_analyst', 'general']

--- python_tutor system prompt (first 300 chars) ---
You are Pyra, an expert Python and AI tutor with 10 years of teaching experience.

YOUR IDENTITY:
- You specialise in Python, machine learning, deep learning, LangChain, and HuggingFace.
- You have taught everyone from absolute beginners to senior engineers.
- You think of yourself as a patient, enc ...



## CELL 5: Few-Shot Prompt Injection

**Few-shot prompting** teaches the model your desired output format by showing it examples *inside the prompt* itself.  
This is different from model fine-tuning — no weights change.  
The model learns in-context: it sees the pattern and continues it.

Best used when:
- You need a very specific output format (JSON, markdown table, ranked list)
- The model's default response style doesn't match your needs
- You want consistent tone or vocabulary

In [13]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

#  Define your few-shot examples 
# These are question→answer pairs that demonstrate exactly how you want
# the model to respond. The more specific and representative, the better.

few_shot_examples = [
    {
        "question": "What is a Python list?",
        "answer": (
            "**Direct answer:** A list is an ordered, mutable collection of items.\n\n"
            "**Think of it like:** A shopping list — items in order, you can add, remove, or change any of them.\n\n"
            "```python\n"
            "# Create a list\n"
            "fruits = ['apple', 'banana', 'cherry']  # ordered by insertion\n\n"
            "# Add an item\n"
            "fruits.append('date')  # adds to the end\n\n"
            "# Change an item (mutable)\n"
            "fruits[0] = 'avocado'  # index 0 = first item\n\n"
            "# Remove an item\n"
            "fruits.remove('banana')  # removes by value\n"
            "```\n\n"
            "**Common mistake:** Confusing `remove()` (by value) with `pop()` (by index).\n\n"
            "What would you like to explore next?"
        ),
    },
    {
        "question": "What is a Python dictionary?",
        "answer": (
            "**Direct answer:** A dictionary is an unordered collection of key-value pairs.\n\n"
            "**Think of it like:** A real dictionary — you look up a word (key) to get its definition (value). Keys must be unique, just like entries in a dictionary.\n\n"
            "```python\n"
            "# Create a dictionary\n"
            "student = {\n"
            "    'name': 'Ali',    # key: 'name', value: 'Ali'\n"
            "    'gpa': 3.8,       # key: 'gpa', value: 3.8\n"
            "    'courses': ['AI', 'Python']  # values can be any type\n"
            "}\n\n"
            "# Access a value\n"
            "print(student['name'])   # 'Ali'\n\n"
            "# Safe access (won't crash if key missing)\n"
            "gpa = student.get('gpa', 0.0)  # returns 0.0 if 'gpa' not found\n"
            "```\n\n"
            "**Common mistake:** Using `dict['missing_key']` raises KeyError. Always use `.get()` when the key might not exist.\n\n"
            "What would you like to explore next?"
        ),
    },
    {
        "question": "What is a lambda function?",
        "answer": (
            "**Direct answer:** A lambda is an anonymous, single-expression function defined in one line.\n\n"
            "**Think of it like:** A sticky note function — written quickly, used once, then discarded. Unlike a proper named function, it has no name and does exactly one thing.\n\n"
            "```python\n"
            "# Regular function\n"
            "def square(x):\n"
            "    return x ** 2\n\n"
            "# Same thing as a lambda\n"
            "square = lambda x: x ** 2  # lambda <args>: <expression>\n\n"
            "# Most useful inline — e.g. sorting by a custom key\n"
            "students = [('Ali', 3.5), ('Sara', 3.9), ('Umar', 2.8)]\n"
            "students.sort(key=lambda s: s[1])  # sort by GPA (index 1)\n"
            "```\n\n"
            "**Common mistake:** Using lambdas for anything complex. If it needs an if-else or multiple lines, write a proper `def` function instead.\n\n"
            "What would you like to explore next?"
        ),
    },
]

#  Build the few-shot prompt template 
# This tells LangChain the format of each example
example_prompt = ChatPromptTemplate.from_messages([
    ("human",     "{question}"),
    ("assistant", "{answer}"),
])

# Wrap all examples into a FewShotChatMessagePromptTemplate
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=few_shot_examples,
)

# Full prompt: system → few-shot examples → actual user question
full_few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system",    get_persona("python_tutor")),
    few_shot_prompt,          # injected examples go here
    ("human",     "{question}"),
])

# Build chain
few_shot_chain = full_few_shot_prompt | llm | StrOutputParser()

#  Test it 
print("=" * 60)
print("FEW-SHOT CHAIN TEST")
print("=" * 60)

test_q = "What is a Python decorator?"
print(f"Question: {test_q}\n")
response = few_shot_chain.invoke({"question": test_q})
print(response)

FEW-SHOT CHAIN TEST
Question: What is a Python decorator?

**Direct answer:** A decorator is a function that takes another function as input and returns a modified version of it.

**Think of it like:** A wrapping paper factory — you put your gift (function) inside, and the factory (decorator) adds extra features (like a ribbon or bow) before giving it back to you.

```python
# A simple decorator
def log_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

# Apply decorator
@log_decorator
def add(a, b):
    return a + b

add(3, 5)  # prints logs + returns 8
```

**Common mistake:** Thinking decorators only work on functions. They can also wrap classes or methods.

What would you like to explore next?



## CELL 6: Chain-of-Thought (CoT) Prompt Mode

**Chain-of-thought prompting** forces the model to reason step by step before stating an answer.  
This dramatically improves accuracy on:
- Logic and math problems
- Multi-step reasoning tasks
- Debugging and root cause analysis
- Any question where jumping to a conclusion is risky

The key phrase is: *"Let's think step by step"* or instructing it to show its reasoning before the final answer.

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#  Chain-of-Thought system prompt 
COT_SYSTEM_PROMPT = """\
You are a rigorous analytical assistant that solves problems through structured reasoning.

MANDATORY REASONING PROTOCOL:
You MUST follow this exact structure for every response. Never skip steps.

## Understanding the problem
(Restate what is being asked in your own words. Identify what is known and what is unknown.)

## Breaking it down
(List the sub-problems or steps you need to address, numbered.)

## Working through each step
(Solve each sub-problem one at a time. Show your reasoning. Question your assumptions.)

## Checking for errors
(Briefly check: Does this make sense? Are there edge cases? Is anything ambiguous?)

## Final answer
(State the clear, direct answer. If it's code, put it in a code block here.)

RULES:
- Never state the final answer before completing all reasoning steps.
- If you catch a mistake mid-way, say 'Wait — let me reconsider.' and correct it.
- If genuinely uncertain, say so in the checking step.
"""

cot_prompt = ChatPromptTemplate.from_messages([
    ("system", COT_SYSTEM_PROMPT),
    ("human",  "{question}"),
])

cot_chain = cot_prompt | llm | StrOutputParser()

#  Test 1: Logic problem 
print("=" * 60)
print("CHAIN-OF-THOUGHT — Logic Problem")
print("=" * 60)

logic_q = (
    "I have a list of numbers: [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]. "
    "I want to find all numbers that appear more than once, "
    "sorted in descending order. What is the result and "
    "what is the most Pythonic way to compute it?"
)
print(f"Q: {logic_q}\n")
print(cot_chain.invoke({"question": logic_q}))

print()

#  Test 2: Debugging problem 
print("=" * 60)
print("CHAIN-OF-THOUGHT — Debugging")
print("=" * 60)

debug_q = """\
This code is supposed to calculate the average of a list, but it gives wrong results. Find all bugs:

```python
def average(numbers):
    total = 0
    for n in numbers:
        total =+ n
    return total / len(numbers)

print(average([10, 20, 30]))  # Expected: 20.0, Got: 30.0
print(average([]))            # Should handle this gracefully
```
"""
print(f"Q: (see above)\n")
print(cot_chain.invoke({"question": debug_q}))

CHAIN-OF-THOUGHT — Logic Problem
Q: I have a list of numbers: [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]. I want to find all numbers that appear more than once, sorted in descending order. What is the result and what is the most Pythonic way to compute it?

## Understanding the problem
We are given a list of integers and need to find all numbers that appear more than once. The result should be sorted in descending order. The task involves identifying duplicates and then sorting them.

## Breaking it down
1. Identify duplicates in the list.
2. Sort the duplicates in descending order.
3. Return the sorted list of duplicates.

## Working through each step
1. **Identify duplicates**: We can use a dictionary to count occurrences of each number in the list.
2. **Filter duplicates**: From the dictionary, extract numbers that have a count greater than 1.
3. **Sort in descending order**: Sort the filtered list of duplicates in descending order.

## Checking for errors
- Ensure that the filtering correctly 


## CELL 7: Conversation Memory

Without memory, every question is treated as if it's the first — the model has no context of what was said before.  

We add memory by maintaining a **message history** and passing the full conversation to the model on each turn.  

Two strategies:
- **Full buffer** — keep every message (good for short sessions)
- **Window buffer** — keep only the last N turns (prevents context overflow)

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from collections import deque

class ConversationMemory:
    """
    Manages chat history for a multi-turn conversation.

    Supports both full-buffer and sliding-window modes.
    Automatically summarizes older turns when the window is exceeded
    (to preserve context without blowing up the context window).
    """

    def __init__(self, window_size: int = 10):
        """
        Parameters
        ----------
        window_size : int
            Maximum number of message PAIRS (human + AI) to keep in context.
            Older messages beyond this window are dropped.
            Set to 0 for unlimited (full buffer).
        """
        self.window_size = window_size
        # Deque: a list that automatically drops from the left when full
        max_msgs = window_size * 2 if window_size > 0 else None
        self.messages = deque(maxlen=max_msgs)
        self.turn_count = 0

    def add_user_message(self, content: str):
        """Record a user message."""
        self.messages.append(HumanMessage(content=content))

    def add_ai_message(self, content: str):
        """Record an AI response."""
        self.messages.append(AIMessage(content=content))
        self.turn_count += 1

    def get_messages(self) -> list:
        """Return the current message history as a list."""
        return list(self.messages)

    def clear(self):
        """Wipe the conversation history."""
        self.messages.clear()
        self.turn_count = 0
        print("Memory cleared.")

    def show(self):
        """Pretty-print the full conversation history."""
        print(f"\n{''*50}")
        print(f"Conversation history ({len(self.messages)} messages, {self.turn_count} turns):")
        print(f"{''*50}")
        for msg in self.messages:
            role  = "You" if isinstance(msg, HumanMessage) else "Bot"
            print(f"{role}: {msg.content[:120]}{'...' if len(msg.content) > 120 else ''}")
        print(f"{''*50}\n")


class MemoryQABot:
    """
    A Q&A bot with conversation memory and a swappable persona.

    Usage
    -----
    bot = MemoryQABot(llm, persona="python_tutor", window_size=6)
    bot.chat("What is a closure?")
    bot.chat("Can you give me another example?")  # bot remembers the previous Q
    """

    def __init__(self, llm, persona: str = "general", window_size: int = 10):
        self.llm      = llm
        self.persona  = persona
        self.memory   = ConversationMemory(window_size=window_size)
        self._build_chain()

    def _build_chain(self):
        """Construct the LangChain LCEL chain with memory placeholder."""
        self.prompt = ChatPromptTemplate.from_messages([
            ("system",    get_persona(self.persona)),
            MessagesPlaceholder(variable_name="history"),  # memory goes here
            ("human",     "{question}"),
        ])
        self.chain = self.prompt | self.llm | StrOutputParser()

    def switch_persona(self, persona: str):
        """Switch to a different persona without losing memory."""
        self.persona = persona
        self._build_chain()
        print(f"Switched to persona: {persona}")

    def chat(self, question: str, verbose: bool = True) -> str:
        """
        Send a question and get a response. Memory is automatically updated.

        Parameters
        ----------
        question : str  — the user's question
        verbose  : bool — if True, print the response

        Returns
        -------
        str — the bot's response
        """
        # Get current history
        history = self.memory.get_messages()

        # Invoke chain
        response = self.chain.invoke({
            "history":  history,
            "question": question,
        })

        # Store both sides of the conversation in memory
        self.memory.add_user_message(question)
        self.memory.add_ai_message(response)

        if verbose:
            print(f"\nYou:  {question}")
            print(f"\nBot:  {response}")
            print(f"\n{''*50}")

        return response


#  Test multi-turn memory 
print("=" * 60)
print("MEMORY BOT — Multi-turn Conversation Test")
print("=" * 60)

bot = MemoryQABot(llm, persona="python_tutor", window_size=8)

# Turn 1 — ask something
bot.chat("What is a Python generator?")

# Turn 2 — refer to the previous answer without restating it
# The bot should know 'it' = generator from memory
bot.chat("Can you show me a real-world example of it?")

# Turn 3 — follow-up that only makes sense with context
bot.chat("How does this compare to a list comprehension in terms of memory usage?")

# Show full conversation history
bot.memory.show()

MEMORY BOT — Multi-turn Conversation Test

You:  What is a Python generator?

Bot:  A Python generator is a special type of iterator that allows you to declare a function that behaves like an iterator, i.e., it can be used in a for loop. Generators are written like regular functions but use the `yield` statement whenever they want to return data. Each time `next()` is called on it, the generator resumes where it left off (it remembers all the data values and which statement was last executed).

**Intuition:** Think of a generator like a coffee machine. You put in the ingredients (call the function), press the button (call `next()`), and it starts brewing (executes the function). When it's ready, you get your coffee (the yielded value). The next time you press the button, it picks up right where it left off, not from the start.

```python
def countdown(n):
    while n > 0:
        yield n  # Yield the current value of n
        n -= 1   # Decrement n

# Using the generator
for num in co


## CELL 8: RAG Setup (Retrieval-Augmented Generation)

**RAG** lets the bot answer questions about *your own documents*,  content it was never trained on.

**How it works:**
1. Load documents (text, PDF, URLs)
2. Split them into chunks
3. Embed each chunk into a vector (a list of numbers capturing meaning)
4. Store embeddings in ChromaDB
5. When a question arrives, embed it and find the most similar chunks
6. Inject those chunks as context into the prompt

Everything — embedding and retrieval — runs locally.

In [16]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
import os, pathlib

#  Step 1: Local Embeddings 
# all-MiniLM-L6-v2 is fast, small (~80MB), and works well for Q&A retrieval.
# Downloaded once, cached locally in ~/.cache/huggingface/
print("Loading embedding model (first run downloads ~80MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},    # change to 'cuda' if you have a GPU
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.")

# Quick test — embed a sentence
test_vector = embeddings.embed_query("What is Python?")
print(f"   Embedding dimension: {len(test_vector)} (each sentence → {len(test_vector)}D vector)")

#  Step 2: Create Sample Documents 
# In real use, load from PDFs, text files, URLs, etc.
# Here we create documents in-memory for demonstration.

sample_documents = [
    Document(
        page_content="""
LangChain is a framework for building applications powered by large language models (LLMs).
It provides abstractions for chaining prompts, managing memory, and integrating with external data sources.
The core concept is a 'chain': a sequence of calls to an LLM, tools, or data retrievers.
LangChain supports Python and JavaScript. The Python package is langchain.
Key components: PromptTemplate, LLM, Chain, Memory, Retriever, Agent, Tool.
LangChain Expression Language (LCEL) uses the | pipe operator to compose chains declaratively.
        """,
        metadata={"source": "langchain_overview.txt", "topic": "LangChain"}
    ),
    Document(
        page_content = """Ollama is a tool for running large language models locally on your own machine. It supports models like llama3, mistral, phi3, gemma2, codellama, and many others. Ollama exposes a REST API on localhost:11434 by default. To pull a model: ollama pull llama3.2 To run interactively: ollama run llama3.2 Ollama handles model quantization, GPU offloading, and memory management automatically. Models are stored in ~/.ollama/models on Linux/Mac and C:\\Users\\<user>\\.ollama on Windows.""",
        metadata={"source": "ollama_guide.txt", "topic": "Ollama"}
    ),
    Document(
        page_content="""
ChromaDB is an open-source embedding database designed for AI applications.
It stores text chunks alongside their vector embeddings for fast similarity search.
ChromaDB runs fully in-memory or on-disk with no external server required.
To persist data to disk: Chroma(persist_directory='./chroma_db', ...)
Similarity search returns the top-k most similar documents to a query vector.
Distance metrics supported: cosine (default), L2, inner product.
ChromaDB integrates natively with LangChain via langchain-chroma.
        """,
        metadata={"source": "chromadb_notes.txt", "topic": "ChromaDB"}
    ),
    Document(
        page_content="""
Retrieval-Augmented Generation (RAG) is a technique that combines document retrieval with text generation.
Instead of relying solely on what the LLM learned during training, RAG fetches relevant documents
at inference time and injects them as context into the prompt.
RAG pipeline: query → embed query → search vector DB → retrieve top-k chunks → build prompt → LLM → answer.
RAG significantly reduces hallucination because the model is grounded in real retrieved text.
Chunking strategy matters: too small loses context, too large dilutes relevance.
Common chunk size: 500–1000 characters with 100–200 character overlap.
        """,
        metadata={"source": "rag_explainer.txt", "topic": "RAG"}
    ),
    Document(
        page_content="""
Prompt engineering is the practice of designing inputs to LLMs to elicit the best outputs.
Key techniques:
- Zero-shot: ask the question with no examples
- Few-shot: provide 2-5 examples of the desired input→output format
- Chain-of-thought: instruct the model to reason step by step before answering
- System prompts: define the model's persona, constraints, and output format
- Self-consistency: generate multiple answers and take the majority vote
Temperature controls randomness: 0.0 = deterministic, 1.0 = creative/random.
Token budget: be concise in prompts — every token costs latency and compute.
        """,
        metadata={"source": "prompt_engineering.txt", "topic": "Prompting"}
    ),
]

#  Step 3: Split into chunks 
# Documents might be long. We split them so each chunk fits in a prompt
# and the retriever returns precise, focused pieces.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,        # characters per chunk
    chunk_overlap=80,      # overlap between consecutive chunks (preserves context)
    separators=["\n\n", "\n", ". ", " "],  # try splitting at paragraph, line, sentence
)

chunks = splitter.split_documents(sample_documents)
print(f"\nSplit {len(sample_documents)} documents → {len(chunks)} chunks")
print(f"Sample chunk:\n  {chunks[0].page_content[:150]}...")
print(f"  Metadata: {chunks[0].metadata}")

#  Step 4: Build ChromaDB vector store 
CHROMA_DIR = "./chroma_qa_bot"   # persisted on disk

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="qa_bot_docs",
)

print(f"\nVector store built. {vector_store._collection.count()} vectors stored.")
print(f"   Persisted to: {pathlib.Path(CHROMA_DIR).absolute()}")

#  Step 5: Test retrieval 
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},    # return top-3 most similar chunks
)

print("\n--- Retrieval Test ---")
test_query   = "How does RAG reduce hallucination?"
retrieved    = retriever.invoke(test_query)
print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved)} chunks:")
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] (source: {doc.metadata['source']})")
    print(f"      {doc.page_content[:100]}...")

Loading embedding model (first run downloads ~80MB)...


C:\Users\Moonwalking\AppData\Local\Temp\ipykernel_18440\4111349020.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8051.09it/s]


Embedding model ready.
   Embedding dimension: 384 (each sentence → 384D vector)

Split 5 documents → 10 chunks
Sample chunk:
  LangChain is a framework for building applications powered by large language models (LLMs).
It provides abstractions for chaining prompts, managing me...
  Metadata: {'source': 'langchain_overview.txt', 'topic': 'LangChain'}

Vector store built. 20 vectors stored.
   Persisted to: C:\Users\Moonwalking\PycharmProjects\Q&A_bot\chroma_qa_bot

--- Retrieval Test ---
Query: 'How does RAG reduce hallucination?'
Retrieved 3 chunks:
  [1] (source: rag_explainer.txt)
      RAG significantly reduces hallucination because the model is grounded in real retrieved text.
Chunki...
  [2] (source: rag_explainer.txt)
      RAG significantly reduces hallucination because the model is grounded in real retrieved text.
Chunki...
  [3] (source: rag_explainer.txt)
      Retrieval-Augmented Generation (RAG) is a technique that combines document retrieval with text gener...


"
## CELL 9: RAG Q&A Chain

Now we connect retrieval to the LLM.  
The chain is: **question → retrieve chunks → format prompt → LLM → answer**.

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

#  RAG System Prompt 
# This tells the model to ONLY use the retrieved context, never make things up.
RAG_SYSTEM_PROMPT = """\
You are a precise technical assistant. You answer questions ONLY using the provided context.

RULES:
1. Base your answer entirely on the CONTEXT section below.
2. If the context does not contain enough information to answer, say:
   'I don't have enough information in my knowledge base to answer that.'
3. Do NOT use prior knowledge or make up information.
4. Cite which source your answer came from (e.g. 'According to rag_explainer.txt...')
5. Be concise. Quote directly from the context when it's clearer than paraphrasing.

CONTEXT:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human",  "{question}"),
])

def format_retrieved_docs(docs) -> str:
    """
    Convert a list of retrieved Documents into a single formatted string
    that we inject into the prompt context.

    We include the source metadata so the model can cite it.
    """
    sections = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        sections.append(f"[Source {i}: {source}]\n{doc.page_content.strip()}")
    return "\n\n".join(sections)

#  Build the RAG chain 
# RunnableParallel runs retrieval and passthrough in parallel,
# then merges results into a dict for the prompt.
rag_chain = (
    RunnableParallel({
        "context":  retriever | format_retrieved_docs,   # retrieve + format
        "question": RunnablePassthrough(),                # pass question through unchanged
    })
    | rag_prompt
    | llm
    | StrOutputParser()
)

#  Test the RAG chain 
rag_questions = [
    "What is LangChain and what are its key components?",
    "How do I run a model with Ollama?",
    "What chunk size should I use for RAG?",
    "What is the difference between zero-shot and few-shot prompting?",
    "What is the capital of France?",   # Not in our docs — should say so
]

print("=" * 60)
print("RAG Q&A CHAIN TESTS")
print("=" * 60)

for q in rag_questions:
    print(f"\nQuestion: {q}")
    answer = rag_chain.invoke(q)
    print(f"Answer:   {answer}")
    print("" * 50)

RAG Q&A CHAIN TESTS

Question: What is LangChain and what are its key components?
Answer:   LangChain is a framework for building applications powered by large language models (LLMs). It provides abstractions for chaining prompts, managing memory, and integrating with external data sources. The core concept is a 'chain': a sequence of calls to an LLM, tools, or data retrievers. LangChain supports Python and JavaScript. The Python package is langchain.

Key components include:
- PromptTemplate
- LLM
- Chain
- Memory
- Retriever
- Agent
- Tool

LangChain Expression Language (LCEL) uses the | pipe operator to compose chains declaratively.


Question: How do I run a model with Ollama?
Answer:   According to ollama_guide.txt, to run a model with Ollama you use the command: ollama run llama3.2


Question: What chunk size should I use for RAG?
Answer:   According to rag_explainer.txt, the common chunk size for RAG is 500–1000 characters with 100–200 character overlap.


Question: What is the 

##  CELL 10: The Full Bot Class (Everything Combined)

This single class unifies:
- Persona (swappable)
- Prompt mode (zero-shot / few-shot / chain-of-thought / RAG)
- Conversation memory
- Source citation
- Conversation logging to JSON

In [18]:
import json
import time
import pathlib
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

class QABot:
    """
    A fully-featured local Q&A bot backed by Ollama.

    Modes
    -----
    'standard'  : System prompt + memory. Good for most conversations.
    'few_shot'  : Injects examples to shape the output format.
    'cot'       : Forces step-by-step reasoning before answering.
    'rag'       : Answers from your own documents using ChromaDB.

    Usage
    -----
    bot = QABot(
        llm=llm,
        persona='python_tutor',
        mode='rag',
        retriever=retriever,
    )
    bot.chat('What is LangChain?')
    bot.switch_mode('cot')
    bot.chat('Debug this function...')
    bot.save_log('session.json')
    """

    VALID_MODES    = {"standard", "few_shot", "cot", "rag"}
    VALID_PERSONAS = set(PERSONAS.keys())

    def __init__(
        self,
        llm,
        persona:     str  = "general",
        mode:        str  = "standard",
        window_size: int  = 10,
        retriever         = None,
        examples:    list = None,
        temperature: float = 0.3,
    ):
        self.llm         = llm
        self.persona     = persona
        self.mode        = mode
        self.retriever   = retriever
        self.examples    = examples or few_shot_examples
        self.memory      = ConversationMemory(window_size=window_size)
        self.log         = []
        self._chain      = None
        self._build_chain()

    #  Chain Construction 

    def _build_chain(self):
        """Rebuild the LCEL chain based on current persona and mode."""
        if self.mode == "rag":
            self._chain = self._build_rag_chain()
        elif self.mode == "few_shot":
            self._chain = self._build_few_shot_chain()
        elif self.mode == "cot":
            self._chain = self._build_cot_chain()
        else:
            self._chain = self._build_standard_chain()

    def _build_standard_chain(self):
        prompt = ChatPromptTemplate.from_messages([
            ("system",  get_persona(self.persona)),
            MessagesPlaceholder(variable_name="history"),
            ("human",   "{question}"),
        ])
        return prompt | self.llm | StrOutputParser()

    def _build_few_shot_chain(self):
        example_prompt = ChatPromptTemplate.from_messages([
            ("human",     "{question}"),
            ("assistant", "{answer}"),
        ])
        fsp = FewShotChatMessagePromptTemplate(
            example_prompt=example_prompt,
            examples=self.examples,
        )
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_persona(self.persona)),
            fsp,
            MessagesPlaceholder(variable_name="history"),
            ("human",  "{question}"),
        ])
        return prompt | self.llm | StrOutputParser()

    def _build_cot_chain(self):
        cot_system = get_persona(self.persona) + "\n\n" + COT_SYSTEM_PROMPT
        prompt = ChatPromptTemplate.from_messages([
            ("system", cot_system),
            MessagesPlaceholder(variable_name="history"),
            ("human",  "{question}"),
        ])
        return prompt | self.llm | StrOutputParser()

    def _build_rag_chain(self):
        if self.retriever is None:
            raise ValueError("RAG mode requires a retriever. Pass retriever=... to QABot.")

        rag_system = f"{get_persona(self.persona)}\n\n{RAG_SYSTEM_PROMPT}"
        prompt = ChatPromptTemplate.from_messages([
            ("system", rag_system),
            MessagesPlaceholder(variable_name="history"),
            ("human",  "{question}"),
        ])

        def retrieve_and_format(inputs):
            """Retrieve docs and inject them into the context slot."""
            docs    = self.retriever.invoke(inputs["question"])
            context = format_retrieved_docs(docs)
            return {
                "context":  context,
                "question": inputs["question"],
                "history":  inputs["history"],
            }

        return retrieve_and_format | prompt | self.llm | StrOutputParser()

    #  Public API 

    def chat(self, question: str, verbose: bool = True) -> str:
        """Send a question to the bot and get a response."""
        history  = self.memory.get_messages()
        start    = time.time()

        inputs = {"question": question, "history": history}
        if self.mode == "rag":
            inputs["context"] = ""   # filled by retrieve_and_format

        response = self._chain.invoke(inputs)
        elapsed  = time.time() - start

        self.memory.add_user_message(question)
        self.memory.add_ai_message(response)

        entry = {
            "timestamp": datetime.now().isoformat(),
            "mode":      self.mode,
            "persona":   self.persona,
            "question":  question,
            "answer":    response,
            "elapsed_s": round(elapsed, 2),
        }
        self.log.append(entry)

        if verbose:
            badge = {"standard": "💬", "few_shot": "📚", "cot": "🧠", "rag": "🔍"}.get(self.mode, "🤖")
            print(f"\n{badge} [{self.mode.upper()} | {self.persona}] ({elapsed:.1f}s)")
            print(f"\n You:  {question}")
            print(f"\n Bot:  {response}")
            print(f"\n{''*60}")

        return response

    def switch_mode(self, mode: str):
        """Switch prompt mode without clearing memory."""
        if mode not in self.VALID_MODES:
            raise ValueError(f"Invalid mode '{mode}'. Choose from: {self.VALID_MODES}")
        self.mode = mode
        self._build_chain()
        print(f"🔄 Mode switched to: {mode}")

    def switch_persona(self, persona: str):
        """Switch persona without clearing memory."""
        if persona not in self.VALID_PERSONAS:
            raise ValueError(f"Invalid persona. Options: {self.VALID_PERSONAS}")
        self.persona = persona
        self._build_chain()
        print(f"Persona switched to: {persona}")

    def add_document(self, text: str, source: str = "manual"):
        """Add a new document to the RAG knowledge base at runtime."""
        if self.retriever is None:
            print("No retriever configured. Switch to RAG mode first.")
            return
        doc    = Document(page_content=text, metadata={"source": source})
        chunks = splitter.split_documents([doc])
        vector_store.add_documents(chunks)
        print(f"Added {len(chunks)} chunk(s) from '{source}' to knowledge base.")

    def clear_memory(self):
        """Clear conversation history."""
        self.memory.clear()

    def show_memory(self):
        """Display conversation history."""
        self.memory.show()

    def save_log(self, path: str = "conversation_log.json"):
        """Save the full conversation log to a JSON file."""
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.log, f, indent=2, ensure_ascii=False)
        print(f"Log saved to {pathlib.Path(path).absolute()} ({len(self.log)} entries)")

    def stats(self):
        """Print session statistics."""
        if not self.log:
            print("No conversations yet.")
            return
        total    = len(self.log)
        avg_time = sum(e["elapsed_s"] for e in self.log) / total
        modes    = {m: sum(1 for e in self.log if e["mode"] == m) for m in self.VALID_MODES}
        print(f"\nSession Stats")
        print(f"   Total turns     : {total}")
        print(f"   Avg response    : {avg_time:.1f}s")
        print(f"   Mode breakdown  : {modes}")
        print(f"   Current persona : {self.persona}")
        print(f"   Current mode    : {self.mode}")


print("QABot class defined.")
print(f"   Personas : {list(PERSONAS.keys())}")
print(f"   Modes    : standard | few_shot | cot | rag")

QABot class defined.
   Personas : ['python_tutor', 'socratic_debugger', 'data_analyst', 'general']
   Modes    : standard | few_shot | cot | rag


## CELL 11: Full Bot Demo (All Modes)

Test every mode and persona combination.

In [19]:
#  Create the bot 
bot = QABot(
    llm=llm,
    persona="python_tutor",
    mode="standard",
    window_size=10,
    retriever=retriever,
)

print("=" * 60)
print("DEMO: Standard mode (python_tutor persona)")
print("=" * 60)
bot.chat("What is the difference between a list and a tuple in Python?")
bot.chat("When should I use one over the other?")   # follow-up using memory

#  Switch to few-shot 
print()
print("=" * 60)
print("DEMO: Few-shot mode (python_tutor persona)")
print("=" * 60)
bot.switch_mode("few_shot")
bot.chat("What is a Python set?")

#  Switch to chain-of-thought 
print()
print("=" * 60)
print("DEMO: Chain-of-thought mode")
print("=" * 60)
bot.switch_mode("cot")
bot.chat(
    "A function returns a list of integers. "
    "I want to remove all duplicates and sort in ascending order. "
    "What is the most Pythonic one-liner?"
)

#  Switch to RAG 
print()
print("=" * 60)
print("DEMO: RAG mode (answers from our documents)")
print("=" * 60)
bot.switch_mode("rag")
bot.chat("What is ChromaDB and how does it store data?")
bot.chat("What does temperature control in a prompt?")

#  Session stats 
bot.stats()
bot.show_memory()

DEMO: Standard mode (python_tutor persona)

💬 [STANDARD | python_tutor] (2.2s)

 You:  What is the difference between a list and a tuple in Python?

 Bot:  A list in Python is a mutable, ordered collection of elements that can be changed after creation, while a tuple is an immutable, ordered collection of elements that cannot be modified once defined. Think of a list as a shopping cart where you can add, remove, or change items, whereas a tuple is like a fixed recipe where the ingredients and their order are set in stone.

```python
# List example
my_list = [1, 2, 3]
my_list[0] = 10  # Changing an element
my_list.append(4)  # Adding an element
print(my_list)  # Output: [10, 2, 3, 4]

# Tuple example
my_tuple = (1, 2, 3)
# my_tuple[0] = 10  # This would raise an error
print(my_tuple)  # Output: (1, 2, 3)
```

**Gotcha**: The main difference is mutability. Lists allow modification, while tuples do not. This makes tuples slightly faster and more memory-efficient for fixed data.



💬 [STAN


## CELL 12: Add Your Own Documents to RAG

Load from a real `.txt` or `.pdf` file and add to the knowledge base.

In [20]:
#  Option A: Add plain text directly 
custom_text = """
The transformer architecture was introduced in the paper 'Attention Is All You Need' (Vaswani et al., 2017).
It replaced recurrent networks (LSTM, GRU) for sequence modeling.
Key components:
- Multi-head self-attention: lets the model attend to different positions of the input simultaneously
- Positional encoding: injects sequence order information since attention is order-agnostic
- Feed-forward layers: applied independently to each position after attention
- Layer normalization and residual connections: stabilise training
The encoder maps input tokens to contextual embeddings.
The decoder generates output tokens auto-regressively.
GPT models use only the decoder. BERT uses only the encoder. T5 uses both.
"""

bot.add_document(custom_text, source="transformer_architecture.txt")

# Test the newly added knowledge
bot.switch_mode("rag")
bot.chat("What did the paper 'Attention Is All You Need' introduce and why was it significant?")

#  Option B: Load from a .txt file 
# Uncomment and edit the path to load your own text file
#
# from langchain_community.document_loaders import TextLoader
# loader = TextLoader("path/to/your/notes.txt", encoding="utf-8")
# docs   = loader.load()
# chunks = splitter.split_documents(docs)
# vector_store.add_documents(chunks)
# print(f"Added {len(chunks)} chunks from your file.")

#  Option C: Load from a PDF 
# Uncomment and edit the path to load a PDF
#
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("path/to/your/paper.pdf")
# docs   = loader.load()         # returns one Document per page
# chunks = splitter.split_documents(docs)
# vector_store.add_documents(chunks)
# print(f"Added {len(chunks)} chunks from PDF.")

Added 2 chunk(s) from 'transformer_architecture.txt' to knowledge base.
🔄 Mode switched to: rag

🔍 [RAG | python_tutor] (2.9s)

 You:  What did the paper 'Attention Is All You Need' introduce and why was it significant?

 Bot:  **Direct answer:** The paper 'Attention Is All You Need' introduced the Transformer architecture, which replaced recurrent models (like LSTM/GRU) for sequence tasks by using self-attention mechanisms. It was significant because it enabled highly parallelizable, state-of-the-art models for tasks like machine translation and language modeling.

**Think of it like:** A new way to process sequences where each word can "pay attention" to other words in the sequence, instead of processing them one-by-one like in RNNs. This allows the model to capture long-range dependencies more effectively.

```python
# Example of self-attention (simplified)
import torch
import torch.nn.functional as F

# Query, Key, Value vectors
Q = torch.rand(3, 4)  # 3 queries, 4-dimensional
K = 

'**Direct answer:** The paper \'Attention Is All You Need\' introduced the Transformer architecture, which replaced recurrent models (like LSTM/GRU) for sequence tasks by using self-attention mechanisms. It was significant because it enabled highly parallelizable, state-of-the-art models for tasks like machine translation and language modeling.\n\n**Think of it like:** A new way to process sequences where each word can "pay attention" to other words in the sequence, instead of processing them one-by-one like in RNNs. This allows the model to capture long-range dependencies more effectively.\n\n```python\n# Example of self-attention (simplified)\nimport torch\nimport torch.nn.functional as F\n\n# Query, Key, Value vectors\nQ = torch.rand(3, 4)  # 3 queries, 4-dimensional\nK = torch.rand(3, 4)  # 3 keys\nV = torch.rand(3, 4)  # 3 values\n\n# Compute attention scores\nscores = torch.matmul(Q, K.T) / torch.sqrt(torch.tensor(4.0))\nattention_weights = F.softmax(scores, dim=-1)\n\n# Weighted

## CELL 13: Load Existing ChromaDB (Across Sessions)

If you restart the kernel, you don't need to rebuild the vector store from scratch.  
ChromaDB persisted it to disk. Load it back in one call.

In [21]:
# Run this cell instead of CELL 8 if you're resuming a session
# and the ChromaDB folder already exists.

CHROMA_DIR = "./chroma_qa_bot"

if pathlib.Path(CHROMA_DIR).exists():
    vector_store_loaded = Chroma(
        persist_directory=CHROMA_DIR,
        collection_name="qa_bot_docs",
        embedding_function=embeddings,
    )
    retriever_loaded = vector_store_loaded.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3},
    )
    print(f"✅ Loaded existing ChromaDB from {CHROMA_DIR}")
    print(f"   Vectors stored: {vector_store_loaded._collection.count()}")

    # Create a fresh bot using the loaded DB
    bot_resumed = QABot(
        llm=llm,
        persona="general",
        mode="rag",
        retriever=retriever_loaded,
    )
    bot_resumed.chat("What is RAG and why does it reduce hallucination?")
else:
    print(f"⚠️  ChromaDB not found at {CHROMA_DIR}. Run CELL 8 first to build it.")

✅ Loaded existing ChromaDB from ./chroma_qa_bot
   Vectors stored: 22

🔍 [RAG | general] (1.1s)

 You:  What is RAG and why does it reduce hallucination?

 Bot:  RAG stands for Retrieval-Augmented Generation. It is a technique that combines document retrieval with text generation. Instead of relying solely on what the LLM learned during training, RAG fetches relevant documents at inference time and injects them as context into the prompt. This grounding in real retrieved text significantly reduces hallucination because the model has direct access to factual information from the documents.

Source: rag_explainer.txt





## CELL 14: Interactive Chat Loop (REPL)

Run this cell to enter a live chat session in the notebook.  
Type your questions. Use these commands:

| Command | Action |
|---|---|
| `!mode standard` | Switch to standard mode |
| `!mode few_shot` | Switch to few-shot mode |
| `!mode cot` | Switch to chain-of-thought mode |
| `!mode rag` | Switch to RAG (document) mode |
| `!persona python_tutor` | Switch persona |
| `!persona socratic_debugger` | Switch persona |
| `!persona data_analyst` | Switch persona |
| `!persona general` | Switch persona |
| `!memory` | Show conversation history |
| `!clear` | Clear conversation memory |
| `!stats` | Show session statistics |
| `!save` | Save log to JSON |
| `!add <text>` | Add text to RAG knowledge base |
| `quit` | Exit the chat |

> **Note:** Input cells block execution in Jupyter. Press Enter to send each message.

In [22]:
def run_chat_loop(bot: QABot):
    """
    Interactive REPL chat loop.

    Reads user input, dispatches to the bot, and handles
    special ! commands for mode/persona switching.
    """
    print("=" * 60)
    print("  Q&A BOT — Interactive Chat")
    print(f"  Model:   {MODEL_NAME}")
    print(f"  Persona: {bot.persona}")
    print(f"  Mode:    {bot.mode}")
    print("=" * 60)
    print("Type your question. Type 'quit' to exit.")
    print("Type '!help' for a list of commands.")
    print()

    COMMANDS_HELP = """
Available commands:
  !mode <standard|few_shot|cot|rag>       Switch prompt mode
  !persona <python_tutor|socratic_debugger|data_analyst|general>  Switch persona
  !memory   Show conversation history
  !clear    Clear conversation memory
  !stats    Session statistics
  !save     Save log to conversation_log.json
  !add <your text here>                   Add text to RAG knowledge base
  quit      Exit the chat
"""

    while True:
        try:
            user_input = input("\n👤 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\nGoodbye!")
            break

        if not user_input:
            continue

        #  Handle commands 
        if user_input.lower() in {"quit", "exit", "q", "bye"}:
            print("\nGoodbye!")
            bot.save_log()
            break

        if user_input == "!help":
            print(COMMANDS_HELP)
            continue

        if user_input.startswith("!mode "):
            mode = user_input[6:].strip()
            try:
                bot.switch_mode(mode)
            except ValueError as e:
                print(f"{e}")
            continue

        if user_input.startswith("!persona "):
            persona = user_input[9:].strip()
            try:
                bot.switch_persona(persona)
            except ValueError as e:
                print(f"{e}")
            continue

        if user_input == "!memory":
            bot.show_memory()
            continue

        if user_input == "!clear":
            bot.clear_memory()
            continue

        if user_input == "!stats":
            bot.stats()
            continue

        if user_input == "!save":
            bot.save_log()
            continue

        if user_input.startswith("!add "):
            text = user_input[5:].strip()
            bot.add_document(text, source="user_input")
            continue

        #  Send to bot 
        bot.chat(user_input)


#  Start the chat 
# Create a fresh bot in RAG mode so it can answer from documents too
chat_bot = QABot(
    llm=llm,
    persona="python_tutor",
    mode="rag",
    window_size=10,
    retriever=retriever,
)

run_chat_loop(chat_bot)

  Q&A BOT — Interactive Chat
  Model:   rnj-1:8b-cloud
  Persona: python_tutor
  Mode:    rag
Type your question. Type 'quit' to exit.
Type '!help' for a list of commands.


🔍 [RAG | python_tutor] (2.6s)

 You:  hey how are you doing?

 Bot:  I'm doing well, thanks for asking! How can I assist you today?


🔄 Mode switched to: standard

Goodbye!
Log saved to C:\Users\Moonwalking\PycharmProjects\Q&A_bot\conversation_log.json (1 entries)


## CELL 15: Save and Review Session Log

In [23]:
# Save the conversation log from the demo bot (CELL 11)
bot.save_log("demo_session_log.json")

# Load and display the log
with open("demo_session_log.json", "r") as f:
    saved_log = json.load(f)

print(f"\n📋 Session Log — {len(saved_log)} entries\n")
print(f"{''*60}")
for entry in saved_log:
    print(f"[{entry['timestamp']}] mode={entry['mode']} | persona={entry['persona']} | {entry['elapsed_s']}s")
    print(f"  Q: {entry['question'][:80]}..." if len(entry['question']) > 80 else f"  Q: {entry['question']}")
    print(f"  A: {entry['answer'][:100]}..." if len(entry['answer']) > 100 else f"  A: {entry['answer']}")
    print(f"{''*60}")

Log saved to C:\Users\Moonwalking\PycharmProjects\Q&A_bot\demo_session_log.json (7 entries)

📋 Session Log — 7 entries


[2026-05-21T18:20:46.479267] mode=standard | persona=python_tutor | 2.22s
  Q: What is the difference between a list and a tuple in Python?
  A: A list in Python is a mutable, ordered collection of elements that can be changed after creation, wh...

[2026-05-21T18:20:49.122424] mode=standard | persona=python_tutor | 2.64s
  Q: When should I use one over the other?
  A: You should use a list when you need a collection of items that can change over time â€” like adding,...

[2026-05-21T18:20:50.741168] mode=few_shot | persona=python_tutor | 1.62s
  Q: What is a Python set?
  A: **Direct answer:** A set is an unordered collection of unique elements.

**Think of it like:** A cla...

[2026-05-21T18:20:52.483342] mode=cot | persona=python_tutor | 1.74s
  Q: A function returns a list of integers. I want to remove all duplicates and sort ...
  A: **Direct answer:** The most 

## CELL 16: Compare Prompt Modes Side-by-Side

Ask the same question in all four modes and see how the response changes.

In [24]:
from langchain_ollama import ChatOllama

COMPARISON_QUESTION = "Why should I use generators instead of lists in Python?"

print("=" * 60)
print(f"PROMPT MODE COMPARISON")
print(f"Question: {COMPARISON_QUESTION}")
print("=" * 60)

# Use a fresh LLM instance for clean comparisons (no temperature carry-over)
llm_compare = ChatOllama(model=MODEL_NAME, temperature=0.1, num_predict=300)

for mode in ["standard", "few_shot", "cot", "rag"]:
    print(f"\n{''*60}")
    print(f"MODE: {mode.upper()}")
    print(f"{''*60}")

    comparison_bot = QABot(
        llm=llm_compare,
        persona="python_tutor",
        mode=mode,
        retriever=retriever,
    )
    response = comparison_bot.chat(COMPARISON_QUESTION, verbose=False)
    elapsed  = comparison_bot.log[-1]["elapsed_s"]
    print(f"({elapsed}s)\n{response}")

PROMPT MODE COMPARISON
Question: Why should I use generators instead of lists in Python?


MODE: STANDARD

(1.62s)
Generators are more memory-efficient than lists because they yield items one at a time and only when requested, whereas lists store all items in memory upfront. This makes generators ideal for large datasets or infinite sequences.

```python
# List example (stores all values in memory)
def list_example():
    return [i * 2 for i in range(1000000)]

# Generator example (yields values one at a time)
def generator_example():
    for i in range(1000000):
        yield i * 2
```

**Gotcha**: Generators can only be iterated over once. If you need to reuse the data, convert it to a list or store it in a data structure that supports multiple iterations.


MODE: FEW_SHOT

(2.19s)
**Direct answer:** Use generators when you need to iterate over a large or infinite sequence without storing all items in memory.

**Think of it like:** A conveyor belt vs. a warehouse. A generator produce

## Summary

You built a fully local, private Q&A bot with:

| Feature | Implementation |
|---|---|
| Local LLM | Ollama (`llama3.2`, `mistral`, `phi3` — your choice) |
| Persona prompts | 4 detailed system prompts shaping identity and tone |
| Few-shot injection | `FewShotChatMessagePromptTemplate` with 3 examples |
| Chain-of-thought | Structured reasoning protocol in the system prompt |
| Conversation memory | Sliding-window `deque` passed as `MessagesPlaceholder` |
| RAG | ChromaDB + HuggingFace `all-MiniLM-L6-v2` embeddings |
| Unified bot class | `QABot` with `switch_mode()`, `switch_persona()`, `add_document()` |
| Interactive REPL | `run_chat_loop()` with `!commands` |
| Session logging | JSON log with timestamps, mode, persona, elapsed time |

### Next steps
- Load real PDFs or your lecture notes into the RAG knowledge base
- Add a Streamlit or Gradio frontend (`pip install streamlit gradio`)
- Try different models: `ollama pull mistral` → change `MODEL_NAME`
- Experiment with `num_predict`, `temperature`, `top_p` in `ChatOllama`
- Implement agent tools (calculator, web search, code execution) with `langchain.agents`